# Baltic Energy News: POS Analysis with Stanza

**Files**
- Input text: `data/baltic_energy.txt`
- Output report: `output/analysis.txt`

Run all cells (outputs should be visible) before exporting to PDF.


**Note:**
- Selected text: Baltic electricity market news article (`data/baltic_energy.txt`).
- Goal: Analyze writing style via POS — tokenize with Stanza, count nouns/verbs/adjectives/adverbs, compute proportions, and summarize results in `output/analysis.txt`.


In [1]:
"""Setup: imports, paths, and Stanza pipeline
- Downloads English tokenizer+POS models if missing
- Builds the processing pipeline
"""
from pathlib import Path
from collections import Counter
import stanza

input_path = Path("data/baltic_energy.txt")
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)
report_path = output_dir / "analysis.txt"

stanza.download("en", processors="tokenize,pos", verbose=False)
nlp = stanza.Pipeline(
    "en",
    processors="tokenize,pos",
    tokenize_no_ssplit=False,
    verbose=False,
)

text = input_path.read_text(encoding="utf-8").strip()
print(f"Loaded {len(text.splitlines())} lines and {len(text.split())} raw tokens from {input_path}")


Loaded 86 lines and 1223 raw tokens from data\baltic_energy.txt


In [2]:
"""Peek at the text so we know what we're analyzing"""
preview_lines = text.splitlines()[:5]
for i, line in enumerate(preview_lines, start=1):
    print(f"{i:02d}: {line}")


01: Electricity consumption in Estonia, Latvia, and Lithuania exceeds production to such an extent that nearly 40 percent of the electricity consumed comes from imports, according to data from the European Network of Transmission System Operators for Electricity (ENTSO-E). By 2035, Eesti Energia predicts growth in both production and consumption, but none of the Baltic countries have made decisions to build new controllable capacities.
02: 
03: Last year, Estonia produced five terawatt-hours of electricity, while consumption was much higher at 8.1 terawatt-hours. In Latvia and Lithuania, consumption also exceeds production—by only 0.5 terawatt-hours in Latvia, but by six terawatt-hours in Lithuania.
04: 
05: Across the three Baltic countries, a total of 16.7 terawatt-hours was produced and 26.3 terawatt-hours was consumed.


In [3]:
"""Run the pipeline and collect POS stats"""
doc = nlp(text)

sent_count = len(doc.sentences)
tokens = [w for s in doc.sentences for w in s.words]
token_count = len(tokens)

pos_counts = Counter(w.xpos for w in tokens)  # Penn Treebank
upos_counts = Counter(w.upos for w in tokens)  # Universal POS

def pos_sum(pos_list):
    return sum(pos_counts.get(tag, 0) for tag in pos_list)

n_nouns = pos_sum(["NN", "NNS", "NNP", "NNPS"])
n_verbs = pos_sum(["VB", "VBD", "VBG", "VBN", "VBP", "VBZ"])
n_adjs = pos_sum(["JJ", "JJR", "JJS"])
n_advs = pos_sum(["RB", "RBR", "RBS"])

noun_pct = n_nouns / token_count if token_count else 0
verb_pct = n_verbs / token_count if token_count else 0
adj_pct = n_adjs / token_count if token_count else 0
adv_pct = n_advs / token_count if token_count else 0

print(f"Sentences: {sent_count}")
print(f"Tokens: {token_count}")
print("-- POS counts (Penn) --")
print({"nouns": n_nouns, "verbs": n_verbs, "adjectives": n_adjs, "adverbs": n_advs})
print("-- POS proportions --")
print({
    "nouns": round(noun_pct, 3),
    "verbs": round(verb_pct, 3),
    "adjectives": round(adj_pct, 3),
    "adverbs": round(adv_pct, 3),
})


Sentences: 58
Tokens: 1476
-- POS counts (Penn) --
{'nouns': 459, 'verbs': 198, 'adjectives': 99, 'adverbs': 56}
-- POS proportions --
{'nouns': 0.311, 'verbs': 0.134, 'adjectives': 0.067, 'adverbs': 0.038}


In [4]:
"""Save a human-readable report to output/analysis.txt"""
lines = [
    f"Input file: {input_path.name}",
    f"Total sentences: {sent_count}",
    f"Total tokens: {token_count}",
    "",
    "POS counts (Penn Treebank):",
    f"  Nouns: {n_nouns}",
    f"  Verbs: {n_verbs}",
    f"  Adjectives: {n_adjs}",
    f"  Adverbs: {n_advs}",
    "",
    "POS proportions (of all tokens):",
    f"  Nouns: {noun_pct:.3f}",
    f"  Verbs: {verb_pct:.3f}",
    f"  Adjectives: {adj_pct:.3f}",
    f"  Adverbs: {adv_pct:.3f}",
    "",
    "Full POS table (Penn):",
]
lines += [f"  {tag}: {count}" for tag, count in pos_counts.most_common()]
lines += ["", "Full POS table (UPOS):"]
lines += [f"  {tag}: {count}" for tag, count in upos_counts.most_common()]

report_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Report saved to {report_path.resolve()}")
print("\nPreview:\n" + "\n".join(lines[:12]))


Report saved to C:\Users\malle\Documents\code\dt-2025-data-visualisation\notebooks\Assignment\output\analysis.txt

Preview:
Input file: baltic_energy.txt
Total sentences: 58
Total tokens: 1476

POS counts (Penn Treebank):
  Nouns: 459
  Verbs: 198
  Adjectives: 99
  Adverbs: 56

POS proportions (of all tokens):
  Nouns: 0.311


### Interpretation
- Higher noun share reflects factual, entity-heavy reporting (countries, energy terms).
- Verb proportion shows the text is action-oriented (imports, exports, forecasts).
- If adjectives/adverbs are relatively low, the style is informative and less opinionated.
- UPOS table can reveal overall balance across content vs. function words.
